# Metapaths in the matching Pathfinder results

A **metapath** is a sequence of **node type — edge type — node type — …**. For example, `Drug — affects — Gene — associated_with — Disease`.

This notebook summarizes **all paths that matched the 26 passing assets**, using the saved Aragorn TEST run from November 7, 2026. It includes every matching rank, not just the first match. It does not include nonmatching paths from those queries or the 53 failed assets. All analysis is local; no queries are sent.

**Counting rules:**
- Node types are the returned knowledge-graph `categories`; edge types are `predicate`. The `biolink:` prefix is hidden for readability. Multiple categories, if present, are kept together in brackets; none is arbitrarily selected.
- Paths are ordered from the query source to its target by graph connectivity, not by the order of edge IDs.
- Parallel edges with the same predicate are collapsed. If a hop has several predicates, we enumerate their combinations. Each metapath is counted **once per returned path**, even if multiple evidence edges produce it.
- The main `metapath` column follows the requested type–predicate–type definition, ignoring edge direction. `directed_metapath` preserves the actual subject/object direction with arrows. Qualifiers, provenance, and nested support graphs are not expanded into new edge types.
- A path is identified by its response file and bound graph ID. A path matching multiple assets is counted only once globally. Metapath percentages can sum to more than 100% because one path can contain several patterns.

Outputs are saved in this notebook. To rerun, select `.local/TestHarness/.venv/bin/python` as the kernel. The previous notebook's `passing_asset_ranks.json` and the saved `test_results` folder are required.

**Folder guide:** notebook outputs go to `../exports/all_passing_assets/`. Shared generated data is in `../data/`, and editable reference pathways are in `../references/`. See `../START_HERE.txt` for the reading and execution order.

This is an optional broader analysis across all 26 passing assets, including gene–disease and disease–disease queries. For the drug–disease focus, use `drug_disease_metapaths.ipynb`.

In [1]:
from pathlib import Path
from collections import defaultdict
from itertools import product
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "test_results/pathfinder-test-full-20261107T192900Z").is_dir())
BASE = ROOT / "pathfiner_aragorn_exp"
DATA = BASE / "data"
REFERENCES = BASE / "references"
OUT = BASE / "exports/all_passing_assets"
OUT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
assets = json.loads((DATA / "passing_asset_ranks.json").read_text())
assert len(assets) == 26 and all(a["status"] == "PASSED" for a in assets)
asset_info = pd.DataFrame([{k: a[k] for k in
    ["case_id", "asset_id", "query", "expected_nodes", "first_matching_rank"]} for a in assets])
asset_info["asset_key"] = asset_info.case_id + "/" + asset_info.asset_id
print(f"Loaded {len(assets)} passing assets.")

Loaded 26 passing assets.


## Reconstruct each path and its metapaths

The saved auxiliary graphs contain parallel evidence edges. We collapse their connectivity to recover the node sequence, and require that every graph forms a single source-to-target chain. If a graph branches or has missing types, the notebook stops instead of guessing. The per-path export retains the original node IDs, names, categories, edge IDs, and graph ID for inspection.

In [2]:
def binding_ids(binding):
    return binding.get("ids", []) if isinstance(binding, dict) else [b["id"] for b in binding]


def short(value):
    return value.removeprefix("biolink:")


def node_type(node):
    categories = sorted(set(node.get("categories") or []))
    assert categories, "Node has no category"
    labels = [short(x) for x in categories]
    return labels[0] if len(labels) == 1 else "[" + " | ".join(labels) + "]"


def ordered_chain(graph, kg, source, target):
    adjacency = defaultdict(set)
    edge_pairs = defaultdict(list)
    for eid in graph["edges"]:
        edge = kg["edges"][eid]
        s, t = edge["subject"], edge["object"]
        assert s != t, "Self-loop in a bound path"
        adjacency[s].add(t)
        adjacency[t].add(s)
        edge_pairs[frozenset((s, t))].append(eid)
    assert source != target and source in adjacency and target in adjacency
    assert {n for n, neighbors in adjacency.items() if len(neighbors) == 1} == {source, target}
    assert all(len(neighbors) == (1 if n in (source, target) else 2)
               for n, neighbors in adjacency.items()), "Path branches"
    sequence = [source]
    while sequence[-1] != target:
        next_nodes = adjacency[sequence[-1]] - set(sequence)
        assert len(next_nodes) == 1, "Ambiguous or broken path"
        sequence.append(next(iter(next_nodes)))
    assert set(sequence) == set(adjacency), "Disconnected edges"
    hops = [sorted(set(edge_pairs[frozenset((s, t))]))
            for s, t in zip(sequence, sequence[1:])]
    return sequence, hops


def metapaths(sequence, hop_edge_ids, kg):
    types = [node_type(kg["nodes"][n]) for n in sequence]
    options = []
    for s, t, edge_ids in zip(sequence, sequence[1:], hop_edge_ids):
        choices = set()
        for eid in edge_ids:
            edge = kg["edges"][eid]
            assert edge.get("predicate"), "Edge has no predicate"
            direction = "forward" if (edge["subject"], edge["object"]) == (s, t) else "reverse"
            choices.add((short(edge["predicate"]), direction))
        options.append(sorted(choices))
    patterns = set()
    for combination in product(*options):
        plain = directed = types[0]
        for (predicate, direction), category in zip(combination, types[1:]):
            plain += f" — {predicate} — {category}"
            directed += (f" —{predicate}→ {category}" if direction == "forward"
                         else f" ←{predicate}— {category}")
        patterns.add((plain, directed))
    return types, patterns


by_response = defaultdict(list)
for asset in assets:
    by_response[asset["response_file"]].append(asset)

path_records, pattern_records, membership_records = [], [], []
for response_file, response_assets in sorted(by_response.items()):
    message = json.loads((ROOT / response_file).read_text())["message"]
    result = message["results"][0]
    kg = message["knowledge_graph"]
    query_paths = message["query_graph"]["paths"]
    seen = {}
    for asset in response_assets:
        for hit in asset["matching_paths"]:
            rank = hit["rank"]
            analysis = result["analyses"][rank - 1]
            bound = {}
            for qpath_id, binding in analysis["path_bindings"].items():
                query_path = query_paths[qpath_id]
                source_ids = binding_ids(result["node_bindings"][query_path["subject"]])
                target_ids = binding_ids(result["node_bindings"][query_path["object"]])
                assert len(source_ids) == len(target_ids) == 1, "Ambiguous endpoints"
                for gid in binding_ids(binding):
                    endpoints = (source_ids[0], target_ids[0])
                    assert gid not in bound or bound[gid] == endpoints
                    bound[gid] = endpoints
            for match in hit["matches"]:
                gid = match["graph_id"]
                assert gid in bound, "Saved match missing from original analysis"
                path_key = response_file + "#" + gid
                membership_records.append({"asset_key": asset["case_id"] + "/" + asset["asset_id"],
                                           "path_key": path_key, "rank": rank})
                if gid in seen:
                    assert seen[gid] == rank, "Graph reused at multiple analysis ranks"
                    continue
                seen[gid] = rank
                graph = message["auxiliary_graphs"][gid]
                sequence, hop_edges = ordered_chain(graph, kg, *bound[gid])
                assert set(match["nodes"]) <= set(sequence)
                types, patterns = metapaths(sequence, hop_edges, kg)
                path_records.append({
                    "path_key": path_key, "case_id": asset["case_id"], "query": asset["query"],
                    "graph_id": gid, "rank": rank, "score": analysis.get("score"),
                    "hop_count": len(hop_edges), "node_type_sequence": " — ".join(types),
                    "node_ids": sequence, "node_names": [kg["nodes"][n].get("name", n) for n in sequence],
                    "node_categories": [kg["nodes"][n]["categories"] for n in sequence],
                    "hop_edge_ids": hop_edges, "response_file": response_file,
                })
                for plain, directed in sorted(patterns):
                    pattern_records.append({"path_key": path_key, "metapath": plain,
                                            "directed_metapath": directed})

paths = pd.DataFrame(path_records)
patterns = pd.DataFrame(pattern_records)
membership = pd.DataFrame(membership_records).drop_duplicates()
assert paths.path_key.is_unique
assert membership.asset_key.nunique() == len(assets)
assert set(membership.path_key) == set(paths.path_key)
assert not patterns.duplicated().any()
plain = patterns[["path_key", "metapath"]].drop_duplicates()
print(f"Validated {len(paths)} unique matching paths across {len(by_response)} saved query responses.")
print(f"Found {plain.metapath.nunique()} metapaths and {patterns.directed_metapath.nunique()} direction-aware metapaths.")

Validated 331 unique matching paths across 11 saved query responses.
Found 1357 metapaths and 1971 direction-aware metapaths.


## Summary of the metapaths

`path_count` is the number of distinct matching returned paths containing a pattern. `asset_count` counts passing assets with at least one such path. Ranks are positions within each original query response; they are not a new ranking across queries. Scores are kept in the path detail export but are not averaged across different queries.

In [3]:
def summarize(pattern_table, label):
    details = pattern_table[["path_key", label]].drop_duplicates().merge(
        paths[["path_key", "case_id", "rank"]], on="path_key", validate="many_to_one")
    summary = details.groupby(label).agg(
        path_count=("path_key", "nunique"), query_count=("case_id", "nunique"),
        best_rank=("rank", "min"), median_rank=("rank", "median"),
    ).reset_index()
    coverage = details[["path_key", label]].merge(membership[["path_key", "asset_key"]], on="path_key")
    asset_counts = coverage.groupby(label).asset_key.nunique().rename("asset_count")
    summary = summary.merge(asset_counts, on=label)
    summary["percent_of_matching_paths"] = (100 * summary.path_count / len(paths)).round(1)
    return summary.sort_values(["path_count", "asset_count", label], ascending=[False, False, True]).reset_index(drop=True)

summary = summarize(plain, "metapath")
directed_summary = summarize(patterns, "directed_metapath")
type_summary = summarize(paths[["path_key", "node_type_sequence"]], "node_type_sequence")

display(pd.DataFrame({"measure": ["Passing assets", "Matching returned paths", "Distinct metapaths",
                                 "Distinct direction-aware metapaths", "Distinct node-type sequences"],
                      "count": [len(assets), len(paths), len(summary), len(directed_summary), len(type_summary)]}))
print("Path lengths (each returned path counted once):")
display(paths.groupby("hop_count").size().rename("path_count").reset_index())
print("20 most frequent metapaths; the CSV contains every pattern:")
with pd.option_context("display.max_colwidth", None):
    display(summary.head(20))

,measure,count
0,Passing assets,26
1,Matching returned paths,331
2,Distinct metapaths,1357
3,Distinct direction-aware metapaths,1971
4,Distinct node-type sequences,61


Path lengths (each returned path counted once):


,hop_count,path_count
0,2,2
1,3,77
2,4,252


20 most frequent metapaths; the CSV contains every pattern:


,metapath,path_count,query_count,best_rank,median_rank,asset_count,percent_of_matching_paths
0,Disease — associated_with — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,66,2,3,172.0,7,19.9
1,Disease — causes — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,56,2,3,179.0,7,16.9
2,Disease — related_to — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,56,2,3,179.0,7,16.9
3,Disease — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,49,2,3,224.0,6,14.8
4,Drug — affects — Gene — has_participant — Pathway — has_participant — Drug — contributes_to — Disease,33,1,93,325.0,1,10.0
5,Drug — affects — Gene — has_participant — Pathway — has_participant — Drug — related_to — Disease,33,1,93,325.0,1,10.0
6,Disease — associated_with — Gene — affects — SmallMolecule — affects — Gene — associated_with — Disease,28,2,2,235.0,5,8.5
7,Disease — predisposes_to_condition — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,28,2,8,146.5,3,8.5
8,Disease — associated_with — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — related_to — Disease,22,2,3,76.0,4,6.6
9,Disease — associated_with — Gene — directly_physically_interacts_with — SmallMolecule — directly_physically_interacts_with — Gene — associated_with — Disease,22,2,2,217.5,3,6.6


## Node-type sequences and edge direction

The first table ignores predicates to show the broader node-type sequences. The second preserves edge direction. A left arrow means the underlying edge points against the source-to-target traversal; it does not change the predicate to its inverse.

In [4]:
with pd.option_context("display.max_colwidth", None):
    display(type_summary.head(15))
    display(directed_summary.head(10))

,node_type_sequence,path_count,query_count,best_rank,median_rank,asset_count,percent_of_matching_paths
0,Disease — Gene — Gene — Gene — Disease,79,2,3,159.0,7,23.9
1,Disease — Gene — SmallMolecule — Gene — Disease,47,2,2,237.0,5,14.2
2,Drug — Gene — Pathway — Drug — Disease,42,2,43,314.0,2,12.7
3,Disease — Gene — Protein — Gene — Disease,19,2,19,176.0,6,5.7
4,Gene — Protein — Gene — Drug,15,1,8,37.0,1,4.5
5,Gene — SmallMolecule — Gene — Drug,14,1,17,85.0,1,4.2
6,Disease — Gene — Gene — Disease,8,2,4,77.5,3,2.4
7,Gene — Drug — Gene — Gene — Disease,8,1,225,376.0,1,2.4
8,Gene — SmallMolecule — Gene — Gene — Disease,8,1,39,378.0,1,2.4
9,Gene — Gene — Cell — Drug,7,1,21,116.0,1,2.1


,directed_metapath,path_count,query_count,best_rank,median_rank,asset_count,percent_of_matching_paths
0,Disease ←associated_with— Gene —physically_interacts_with→ Gene ←physically_interacts_with— Gene —associated_with→ Disease,40,2,3,244.0,7,12.1
1,Disease ←causes— Gene —physically_interacts_with→ Gene ←physically_interacts_with— Gene —associated_with→ Disease,38,2,3,244.0,7,11.5
2,Disease ←related_to— Gene —physically_interacts_with→ Gene ←physically_interacts_with— Gene —associated_with→ Disease,38,2,3,244.0,7,11.5
3,Disease ←affects— Gene —physically_interacts_with→ Gene ←physically_interacts_with— Gene —associated_with→ Disease,34,2,3,262.5,6,10.3
4,Drug —affects→ Gene ←has_participant— Pathway —has_participant→ Drug —contributes_to→ Disease,33,1,93,325.0,1,10.0
5,Drug —affects→ Gene ←has_participant— Pathway —has_participant→ Drug —related_to→ Disease,33,1,93,325.0,1,10.0
6,Disease ←associated_with— Gene ←affects— SmallMolecule —affects→ Gene —associated_with→ Disease,28,2,2,235.0,5,8.5
7,Disease ←associated_with— Gene —physically_interacts_with→ Gene —physically_interacts_with→ Gene —associated_with→ Disease,22,2,5,104.5,6,6.6
8,Disease ←associated_with— Gene ←directly_physically_interacts_with— SmallMolecule —directly_physically_interacts_with→ Gene —associated_with→ Disease,22,2,2,217.5,3,6.6
9,Disease ←associated_with— Gene ←affects— SmallMolecule —affects→ Gene —affects→ Disease,21,2,2,248.0,4,6.3


## Summary for each of the 26 assets

An asset may have many matching paths and many metapaths. The table lists its most frequent metapath and how many of its paths contain it. Ties are broken alphabetically for display only. The per-asset CSV includes every pattern.

In [5]:
asset_patterns = membership.merge(plain, on="path_key").drop_duplicates(["asset_key", "path_key", "metapath"])
per_asset_metapaths = asset_patterns.groupby(["asset_key", "metapath"]).agg(
    path_count=("path_key", "nunique"), best_rank=("rank", "min"), median_rank=("rank", "median")
).reset_index()
per_asset_metapaths = per_asset_metapaths.merge(asset_info, on="asset_key", validate="many_to_one")
per_asset_metapaths = per_asset_metapaths.sort_values(
    ["asset_key", "path_count", "metapath"], ascending=[True, False, True])
counts = asset_patterns.groupby("asset_key").agg(
    matching_paths=("path_key", "nunique"), distinct_metapaths=("metapath", "nunique"))
top = per_asset_metapaths.drop_duplicates("asset_key")[["asset_key", "metapath", "path_count"]].rename(
    columns={"metapath": "most_frequent_metapath", "path_count": "paths_with_top_metapath"})
asset_summary = asset_info.merge(counts, on="asset_key").merge(top, on="asset_key")
assert len(asset_summary) == 26
assert all(asset_summary.set_index("asset_key").matching_paths[a["case_id"] + "/" + a["asset_id"]]
           == a["matching_path_count"] for a in assets)
with pd.option_context("display.max_rows", 30, "display.max_colwidth", None):
    display(asset_summary[["asset_id", "query", "expected_nodes", "matching_paths", "distinct_metapaths",
                           "most_frequent_metapath", "paths_with_top_metapath"]])

,asset_id,query,expected_nodes,matching_paths,distinct_metapaths,most_frequent_metapath,paths_with_top_metapath
0,Asset_2,Imatinib → Asthma,KIT,5,40,Drug — affects — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,2
1,Asset_23,Crohn's Disease → Parkinson's Disease,LRRK2,58,185,Disease — associated_with — Gene — directly_physically_interacts_with — SmallMolecule — directly_physically_interacts_with — Gene — affects — Disease,18
2,Asset_37,IFNG → tryptophan,IDO1,50,80,Gene — affects — Protein — affects — Gene — affects — Drug,15
3,Asset_79,Parkinson's Disease → melanoma,LRRK2,28,160,Disease — affects — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — associated_with — Disease,12
4,Asset_86,Parkinson's Disease → melanoma,MC1R,5,138,Disease — correlated_with — SmallMolecule — affects — Gene — directly_physically_interacts_with — Gene — associated_with — Disease,2
5,Asset_40,IFNG → tryptophan,T Cell,7,27,Gene — affects — Gene — interacts_with — Cell — affects — Drug,5
6,Asset_5,Imatinib → Asthma,Mast Cell,5,22,Drug — affects — Cell — interacts_with — Cell — interacts_with — Cell — disrupts — Disease,1
7,Asset_25,SLC6A20 → COVID-19,ACE2,23,60,Gene — affects — SmallMolecule — affects — Gene — physically_interacts_with — Gene — affects — Disease,7
8,Asset_22,Crohn's Disease → Parkinson's Disease,PARK7,15,92,Disease — associated_with — Gene — physically_interacts_with — Gene — physically_interacts_with — Gene — affects — Disease,9
9,Asset_83,Parkinson's Disease → melanoma,HTRA2,1,2,Disease — causes — Gene — physically_interacts_with — Gene — associated_with — Disease,1


## Inspect one asset

Change the case and asset IDs to inspect another passing asset. This shows all metapaths and the concrete nodes in each matching path. Multiple predicate patterns can share the same node sequence.

In [6]:
asset_key = "TestCase_0/Asset_2"  # Imatinib → asthma, through KIT
print(asset_info.set_index("asset_key").loc[asset_key, ["query", "expected_nodes"]].to_string())
with pd.option_context("display.max_colwidth", None, "display.max_rows", 100):
    display(per_asset_metapaths.loc[per_asset_metapaths.asset_key == asset_key,
                                   ["metapath", "path_count", "best_rank", "median_rank"]])
    selected_paths = paths[paths.path_key.isin(membership.loc[membership.asset_key == asset_key, "path_key"])]
    display(selected_paths[["graph_id", "rank", "node_names", "node_type_sequence"]].sort_values("rank"))

query             Imatinib → Asthma
expected_nodes                  KIT


,metapath,path_count,best_rank,median_rank
0,Drug — affects — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,2,44,105.5
10,Drug — coexists_with — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,2,44,105.5
20,Drug — directly_physically_interacts_with — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,2,44,105.5
30,Drug — interacts_with — Gene — actively_involved_in — BiologicalProcess — genetic_association — Disease,2,44,105.5
1,Drug — affects — Gene — associated_with — Disease — affects — Disease,1,2,2.0
2,Drug — affects — Gene — associated_with — Disease — causes — Disease,1,2,2.0
3,Drug — affects — Gene — associated_with — Disease — coexists_with — Disease,1,2,2.0
4,Drug — affects — Gene — directly_physically_interacts_with — Gene — condition_associated_with_gene — Disease,1,126,126.0
5,Drug — affects — Gene — directly_physically_interacts_with — Gene — gene_associated_with_condition — Disease,1,126,126.0
6,Drug — affects — Gene — physically_interacts_with — Gene — condition_associated_with_gene — Disease,1,28,28.0


,graph_id,rank,node_names,node_type_sequence
0,p_415,2,"[Imatinib, KIT, hypereosinophilic syndrome, asthma]",Drug — Gene — Disease — Disease
1,p_8304,28,"[Imatinib, KIT, TXK, asthma]",Drug — Gene — Gene — Disease
2,p_8418,44,"[Imatinib, KIT, positive regulation of dendritic cell cytokine production, asthma]",Drug — Gene — BiologicalProcess — Disease
3,p_4190,126,"[Imatinib, KIT, SLA, asthma]",Drug — Gene — Gene — Disease
4,p_9067,167,"[Imatinib, KIT, primordial germ cell migration, asthma]",Drug — Gene — BiologicalProcess — Disease


## Save the tables

The JSON includes complete path IDs, node IDs/types, edge IDs per hop, and the original response file. The mapping CSV connects each pattern to a returned path and its matching assets, so the aggregate counts can be traced back to the saved responses.

In [7]:
summary.to_csv(OUT / "metapath_summary.csv", index=False)
directed_summary.to_csv(OUT / "metapath_directed_summary.csv", index=False)
type_summary.to_csv(OUT / "metapath_node_type_summary.csv", index=False)
asset_summary.to_csv(OUT / "metapath_asset_summary.csv", index=False)
per_asset_metapaths.to_csv(OUT / "metapaths_per_asset.csv", index=False)
patterns.merge(membership, on="path_key").to_csv(OUT / "metapath_path_mapping.csv", index=False)
paths.to_json(OUT / "metapath_path_details.json", orient="records", indent=2)
print("Saved six CSV tables and metapath_path_details.json in", OUT)


Saved six CSV tables and metapath_path_details.json in /Users/carolinagonzalez/shepherd-new-pathfinder/pathfiner_aragorn_exp/exports/all_passing_assets
